In [2]:
import os
import sys
import pandas as pd
import time

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path: sys.path.append(module_path)
from backup_playlists import update_track_db

import ytmusicapi as ytmusicapi
print(f'Using ytmusicapi version: {ytmusicapi.__version__}')

HEADER_FILE='../headers_auth.json'
print(f'Using header file: {HEADER_FILE}')
yt = ytmusicapi.YTMusic(HEADER_FILE)


YTMUSIC_DB_TSV='../playlists/_tracks_db.tsv'
YTMUSIC_DB_VALID_TSV='../playlists/_tracks_db_valid.tsv' 
YTMUSIC_DB_INVALID_TSV='../playlists/_tracks_db_invalid.tsv'

print(f'Using yt db file: {YTMUSIC_DB_TSV}')
yt_db = pd.read_csv(YTMUSIC_DB_TSV, sep='\t', index_col=0)
print(f"Loaded {len(yt_db)} ytmusic db entries.")


Using ytmusicapi version: 0.24.0
Using header file: ../headers_auth.json
Using yt db file: ../playlists/_tracks_db.tsv
Loaded 132198 ytmusic db entries.


## Expire vIds that do ont exist in yt api

In [6]:
import os
import sys
import pandas as pd
import time

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path: sys.path.append(module_path)
from backup_playlists import update_track_db

import ytmusicapi as ytmusicapi

HEADER_FILE='../headers_auth.json'
YTMUSIC_DB_TSV='../playlists/_tracks_db.tsv'
YTMUSIC_DB_VALID_TSV='../playlists/_tracks_db_valid.tsv' 
YTMUSIC_DB_INVALID_TSV='../playlists/_tracks_db_invalid.tsv'

OVERWRITE_DB_THRESH = 90
PL_NAME = 'zzzz_tmp'
PL_DESC = 'for backlogging valid video ids'
PL_SIZE = 1000
VERBOSE = False
START_AT = PL_SIZE
SLEEP_TIME=2
assert START_AT >= PL_SIZE
valid_id_set = set()
invalid_id_set = set()

print(f'Using ytmusicapi version: {ytmusicapi.__version__}')
print(f'Using header file: {HEADER_FILE}')

yt = ytmusicapi.YTMusic(HEADER_FILE)
print(f'Using yt db file: {YTMUSIC_DB_TSV}')
yt_db = pd.read_csv(YTMUSIC_DB_TSV, sep='\t', index_col=0, dtype=object)
print(f"Loaded {len(yt_db)} ytmusic db entries.")

yt_db_last_invalid = pd.read_csv(YTMUSIC_DB_INVALID_TSV, sep='\t', index_col=0, dtype=object)
print(f"Loaded {len(yt_db_last_invalid)} previously invalid ytmusic db entries.")

for i in range(START_AT, len(yt_db), PL_SIZE):
    try:
        pl_vids = list(yt_db.index[i-PL_SIZE:i])
        pl_id = yt.create_playlist(PL_NAME, PL_DESC, video_ids=pl_vids)
        if VERBOSE:
            print(f'{PL_SIZE} tracks added to {PL_NAME} ({pl_id})')
        time.sleep(SLEEP_TIME)
        pl_res = yt.get_playlist(pl_id, limit=PL_SIZE)
        valid_ids = set()
        for t in pl_res['tracks']:
            valid_ids.add(t['videoId'])
        invalid_ids = set(pl_vids) - valid_ids
        valid_id_set.update(valid_ids)
        invalid_id_set.update(invalid_ids)
        perc_valid = round(100.0*len(valid_id_set)/i)
        print(f'({i//PL_SIZE}/{len(yt_db)//PL_SIZE}): {len(invalid_ids)}',
              f'are invalid tracks, {len(invalid_id_set)}', 
              f'invalid ({perc_valid}% valid) so far')
        time.sleep(SLEEP_TIME)
        yt.delete_playlist(pl_id)
    except Exception as e:
        print(f'Failed on index {i-PL_SIZE}:{i}')
        print(e)


## Save results
yt_db_valid = yt_db.loc[yt_db.index.isin(valid_id_set)]
yt_db_invalid = yt_db.loc[ yt_db.index.isin(invalid_id_set)]

## Valid Tracks
perc_valid = round(100.*len(yt_db_valid)/len(yt_db), 1)
print(f'{len(yt_db_valid)} of {len(yt_db)} ({perc_valid}%)', 
      f'tracks are valid, {len(yt_db_invalid)} invalid')
if perc_valid > OVERWRITE_DB_THRESH:
      print(f'Replacing {YTMUSIC_DB_TSV} with valid only entries', 
            f'(--{len(yt_db)-len(yt_db_valid)})')
      yt_db_valid.to_csv(YTMUSIC_DB_TSV, header=True, index=True, sep='\t')
else:
      print(f'Less than {OVERWRITE_DB_THRESH}% are valid ({perc_valid})', 
            f'Not replacing {YTMUSIC_DB_TSV}, saving valid db as', 
            f'{YTMUSIC_DB_VALID_TSV} with valid only entries')
      yt_db_valid.to_csv(YTMUSIC_DB_VALID_TSV, header=True, index=True, sep='\t')

## Invalid Tracks
unique_new_keys = frozenset(yt_db_last_invalid.index) & frozenset(yt_db_invalid.index)
if len(unique_new_keys) == 0:
      print(f'No new invalid tracks to add to {YTMUSIC_DB_INVALID_TSV}')
else:
      yt_db_invalid_add = yt_db_invalid.loc[yt_db_invalid.index.isin(unique_new_keys)]
      yt_db_invalid = pd.concat([yt_db_last_invalid, yt_db_invalid_add])
      print(f'Old invalid db had {len(yt_db_last_invalid)} entries, current has {len(yt_db_invalid)}',
            f'entries (appended {len(unique_new_keys)} to {YTMUSIC_DB_INVALID_TSV})')
      yt_db_invalid.to_csv(YTMUSIC_DB_INVALID_TSV, header=True, index=True, sep='\t')


(1/132): 0 are invalid tracks, 0 invalid (100% valid) so far
(2/132): 5 are invalid tracks, 5 invalid (100% valid) so far
(3/132): 1 are invalid tracks, 6 invalid (100% valid) so far
(4/132): 0 are invalid tracks, 6 invalid (100% valid) so far
(5/132): 1 are invalid tracks, 7 invalid (100% valid) so far
(6/132): 8 are invalid tracks, 15 invalid (100% valid) so far
(7/132): 0 are invalid tracks, 15 invalid (100% valid) so far
(8/132): 2 are invalid tracks, 17 invalid (100% valid) so far
(9/132): 0 are invalid tracks, 17 invalid (100% valid) so far
(10/132): 0 are invalid tracks, 17 invalid (100% valid) so far
(11/132): 3 are invalid tracks, 20 invalid (100% valid) so far
(12/132): 0 are invalid tracks, 20 invalid (100% valid) so far
(13/132): 1 are invalid tracks, 21 invalid (100% valid) so far
(14/132): 0 are invalid tracks, 21 invalid (100% valid) so far
(15/132): 0 are invalid tracks, 21 invalid (100% valid) so far
(16/132): 1 are invalid tracks, 22 invalid (100% valid) so far
(17/13

131730 of 132198 (99.6%) tracks are valid, 271 invalid
Replacing ../playlists/_tracks_db.tsv with valid only entries (--468)
Old invalid db had 12644 entries, current has 12646 entries (appended 2 to ../playlists/_tracks_db_invalid.tsv)


Using yt db file: ../playlists/_tracks_db.tsv
Loaded 132198 ytmusic db entries.
Loaded 12644 previously invalid ytmusic db entries.


In [ ]:
# TODO refactor above code to script, then remove cells (leave whats below, maybe rename notebook)

## Get missing (nan) album info for yt_db

In [2]:
# Update album fields if Nan
check_cols = [ 'likeStatus', 'albumArtist', 'albumTrackCount',  'albumDuration', 'albumYear', 'albumType']#'duration', 'averageRating',  'viewCount','release']

print(f"Looking for nans in all columns except: {set(yt_db.columns) - set(check_cols)}")


yt_db_has_nan = yt_db[yt_db[check_cols].isnull().any(axis=1)]
yt_db_no_nan =  yt_db.loc[frozenset(yt_db.index) - frozenset(yt_db_has_nan.index)]

print(f"{len(yt_db_has_nan)} of {len(yt_db)} entries have a nan column and will be replaced, {len(yt_db_no_nan)} are ok")

new_yt_db = update_track_db(yt, yt_db_no_nan, yt_db_has_nan)
new_yt_db_has_nan = new_yt_db[new_yt_db[check_cols].isnull().any(axis=1)]
print(f"{len(new_yt_db_has_nan)} of {len(new_yt_db)} entries have a nan column")
new_yt_db.to_csv('_tracks_db__updated_albums.tsv', sep='\t', header=True)



Looking for nans in all columns except: {'keywords', 'release', 'duration', 'album', 'isExplicit', 'title', 'artistId', 'artist', 'duration_seconds', 'albumId', 'isAvailable', 'viewCount', 'playlists', 'averageRating'}
7945 of 114810 entries have a nan column and will be replaced, 106865 are ok
Track database has 106865 tracks, found 7945 unique new tracks

Skipping privately owned track for row: !!! - Slyd
(1/7945): !!! - THR!!!ER - Slyd -> LIKE | nan (nan)

Skipping privately owned track for row: $yrup & Hush - Fox
(2/7945): $yrup & Hush - $yrup & Hush (Smash Bros) - Fox -> LIKE | nan (nan)

Skipping privately owned track for row: $yurp.x.Divine Elite.x.Hush - Luigi
(3/7945): $yurp.x.Divine Elite.x.Hush - $yrup & Hush - Luigi -> LIKE | nan (nan)

D ERROR: row["artistId"] not a str for row:: - - -
(4/7945): - - - - - -> nan | nan (nan)

D ERROR: row["artistId"] not a str for row:: 1.9.9.9, Ghostface Playa, Pharmacist - LIVIN' LAVISH
(5/7945): 1.9.9.9, Ghostface Playa, Pharmacist - LIV

In [ ]:
# # untested other track nan fields
# check_cols = [ 'duration', 'averageRating',  'viewCount','release']

# print(f"Looking for nans in all columns except: {set(yt_db.columns) - set(check_cols)}")


# yt_db_has_nan = yt_db[yt_db[check_cols].isnull().any(axis=1)]
# yt_db_no_nan =  yt_db.loc[frozenset(yt_db.index) - frozenset(yt_db_has_nan.index)]

# print(f"{len(yt_db_has_nan)} of {len(yt_db)} entries have a nan column and will be replaced, {len(yt_db_no_nan)} are ok")

# new_yt_db = update_track_db(yt, yt_db_no_nan, yt_db_has_nan)
# new_yt_db_has_nan = new_yt_db[new_yt_db[check_cols].isnull().any(axis=1)]
# print(f"{len(new_yt_db_has_nan)} of {len(new_yt_db)} entries have a nan column")
# new_yt_db.to_csv('_tracks_db__updated_albums.tsv', sep='\t', header=True)



## Like tracks in like_tsv (but not in yt_db) and make playlist with them

In [27]:
# Like and add to playlist tracks in like tsv but not in db
yt_db_liked = yt_db[yt_db.likeStatus == 'LIKE']
print(f"liked {len(yt_db_liked)} in track_db")

YTMUSIC_LIKE_TSV='../playlists/_liked_tracks.tsv'
tsv_liked = pd.read_csv(YTMUSIC_LIKE_TSV, sep='\t', index_col=0)
print(f"liked {len(tsv_liked)} in like track tsv")

tsv_like_only = frozenset(tsv_liked.index) - frozenset(yt_db_liked.index)
db_liked_only = frozenset(yt_db_liked.index) - frozenset(tsv_liked.index)
print(f'{len(tsv_like_only)} only in tsv and {len(db_liked_only)} only in db')
for vid in tsv_like_only:
    yt.rate_song(vid)
# yt.create_playlist('_tsv_likes', description='tmp', video_ids=list(tsv_like_only))

not_in_db = []
for vid in tsv_like_only:
    if not vid in yt_db.index:
        not_in_db.append(vid)
    else:
        print('manually update yt_db to like for:', vid)


liked 27237 in track_db
liked 30214 in like track tsv
2991 only in tsv and 14 only in db
manually update yt_db to like for: Ali1JQTJQsI
manually update yt_db to like for: yaRSO7uXciY
manually update yt_db to like for: W3dhyvaxPT0
manually update yt_db to like for: p8owaxpDQy8
manually update yt_db to like for: 1v_gWsbPcQk
manually update yt_db to like for: -NPV1I-L4tQ
manually update yt_db to like for: XPfn7kdTJYs
manually update yt_db to like for: qO5ITgzJO2A
